# LIBS Calibration Diagnostic Inspector (R13.3, AC13)
Interactive diagnostic tool for exploring intermediate calibration dumps under `temp/calibration_debug/<materialFamily>/`.

### Features:
- **Family & Material Selectors:** Interactive dropdowns to toggle between calibrated families and reference materials.
- **Morphological Opening Baseline Correction:** Configurable structuring element size (default 25 channels / ~6.75 nm) to strip broad continuum background while preserving sharp atomic emission peaks. Fully interactive slider control with optional continuum trace visualization.
- **Spectral Fit Viewer:** Plotly zoom/pan overlay of measured vs. synthetic spectra with ROI window boundaries, $1\sigma$ measurement ribbon, line species annotations, and secondary-axis instrument response curve $R(\lambda)$.
- **Solver Convergence:** Evolution of objective function $\Phi$ and B-spline response coefficients $\beta$ across alternating iterations.
- **Element Parity Chart:** Bar chart of $\log_{10}(\text{synthetic}/\text{measured})$ across certified elements with bias threshold reference bands ($\pm 0.10$, $\pm 0.30$).
- **Dynamic 5-Point Response & Baseline Tuner:** Interactive sliders for B-spline coefficients $\beta_1 \dots \beta_5$ AND morphological opening size $W$, with real-time recalculation of the synthetic spectrum, $R^2$, and reduced $\chi^2$ metrics.

In [ ]:
# Initialize environment and dependencies
import sys
from pathlib import Path

# Ensure scripts directory is on PATH
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.calibration_inspector import (
    launch_dashboard,
    create_interactive_response_tuner,
    plot_spectral_overlay,
    plot_solver_convergence,
    plot_element_parity,
    load_debug_data,
    list_families,
    list_materials_for_family,
    compute_bspline_basis,
    compute_response_from_beta,
    apply_morphological_baseline_correction
)
print("Calibration Inspector loaded successfully!")

## 1. Full Calibration Diagnostics Dashboard

The dashboard below displays:
1. **Spectral Overlay:** Baseline-corrected measured spectrum (blue) with $1\sigma$ error ribbon, synthetic fit (orange dashed), and spectrometer response curve $R(\lambda)$ (purple dash-dotted line on the right-hand axis).
2. **Baseline Struct Size Slider (default 25):** Morphological opening size to remove continuum background. Set to 1 to view raw uncorrected spectrum.
3. **Show Baseline Checkbox:** Check to display the estimated continuum baseline as a dotted gray trace.
4. **Solver Convergence Trace:** Optimization trajectory for objective $\Phi$ and spline knots $\beta$.
5. **Element Parity Chart:** Relative balance across certified elements.

In [ ]:
# Launch Interactive Dashboard with Baseline Size slider and Response Curve overlay
launch_dashboard()

## 2. Dynamic 5-Point Response & Baseline Tuner

### Morphological Opening Baseline Correction
In laser plasma spectroscopy, the raw measured spectrum contains a continuous background pedestal (Bremsstrahlung radiation and recombining electron-ion emission) under the atomic lines.
Morphological opening applies an erosion followed by a dilation using a flat horizontal structuring element of size $W$ channels:
$$\text{Baseline}(\lambda) = \left(y_{\text{raw}} \circ S_W\right)(\lambda) = \text{dilation}\Big(\text{erosion}\big(y_{\text{raw}}, W\big), W\Big)$$
$$y_{\text{clean}}(\lambda) = \max\left(y_{\text{raw}}(\lambda) - \text{Baseline}(\lambda), 0.0\right)$$

Because atomic emission peaks are narrow (FWHM $\approx 2-4$ pixels $\approx 0.6$ nm), a structuring element of **$W=25$ pixels** ($\\approx 6.75$ nm) acts as a high-pass filter: it rolls smoothly underneath the peaks without clipping their heights, eliminating the baseline offset between lines.

### Spectrometer Response B-Spline Knobs ($\beta_1 \dots \beta_5$)
The response curve $R(\lambda)$ models spectrometer efficiency across wavelengths using a 5-knot cubic B-spline:
$$\ln R(\lambda) = \sum_{j=1}^5 \beta_j B_j(\lambda) \implies R(\lambda) = \exp\left( \sum_{j=1}^5 \beta_j B_j(\lambda) \right)$$

- **$\beta_1$ (UV band, ~340–380 nm):** Controls UV detector sensitivity.
- **$\beta_2$ (Blue band, ~424 nm):** Controls blue-region efficiency.
- **$\beta_3$ (Cyan band, ~479 nm):** Controls mid-visible throughput.
- **$\beta_4$ (Green band, ~534 nm):** Controls green-visible detector response.
- **$\beta_5$ (NIR band, ~580+ nm):** Controls yellow, red, and near-infrared sensitivity.

Use the sliders below to adjust the baseline structuring element size and the 5 spline knobs. Watch the clean measured spectrum, synthetic fit, and live $R^2$ / Reduced $\chi^2$ metrics update in real time!

In [ ]:
# Launch the Dynamic Response & Baseline Tuner (default baseline size = 25)
create_interactive_response_tuner("Ferritic_Stainless_Steel", "469", default_struct_size=25)

## 3. Manual Inspection & Custom Visualization
You can also extract and customize individual plots directly using the helper functions below.

In [ ]:
# Example: Manually load, baseline-correct, and display a specific material
families = list_families()
if families:
    fam = families[0]
    mats = list_materials_for_family(fam)
    if mats:
        mat = mats[0]
        cond_data, rois_data, trace_data, fit_data = load_debug_data(fam, mat)
        
        # Evaluate response curve from trace
        wls = cond_data.get("wavelengths")
        beta_hist = trace_data.get("beta", [])
        resp_curve = compute_response_from_beta(wls, rois_data, beta_hist[-1]) if (beta_hist and wls) else None
        
        # Plot overlay with default morphological opening baseline size 25
        fig_spec = plot_spectral_overlay(
            cond_data, rois_data, fit_data,
            title=f"Spectrum: {fam} / {mat}",
            response_curve=resp_curve,
            baseline_struct_size=25,
            show_baseline=True
        )
        fig_spec.show()
        
        fig_conv = plot_solver_convergence(trace_data, title=f"Solver Convergence: {fam}")
        fig_conv.show()
        
        fig_par = plot_element_parity(fit_data, title=f"Element Parity: {fam} / {mat}")
        fig_par.show()